# Multimodal RAG — Data Exploration

Notebook for prototyping and inspecting the document processing pipeline.

In [ ]:
import sys
sys.path.insert(0, "..")

from src.document.parser import pdf_to_images
from src.document.layout import LayoutDetector
from PIL import Image
import matplotlib.pyplot as plt

## 1. Parse a sample PDF

In [ ]:
# Replace with your PDF path
PDF_PATH = "../data/raw/sample.pdf"

pages = pdf_to_images(PDF_PATH, dpi=200)
print(f"Pages: {len(pages)}")

fig, axes = plt.subplots(1, min(3, len(pages)), figsize=(18, 8))
if len(pages) == 1:
    axes = [axes]
for ax, img in zip(axes, pages[:3]):
    ax.imshow(img)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2. Layout detection

In [ ]:
detector = LayoutDetector()
regions = detector.process_pages(pages)
print(f"Detected {len(regions)} regions")

for r in regions[:10]:
    print(f"  Page {r.page_index}, {r.label} (conf={r.confidence:.2f}), bbox={r.bbox}")

In [ ]:
# Visualize regions on the first page
from PIL import ImageDraw

page = pages[0].copy()
draw = ImageDraw.Draw(page)

colors = {"Table": "red", "Picture": "blue", "Text": "green", "Title": "orange", "Section-header": "purple"}
for r in regions:
    if r.page_index == 0:
        color = colors.get(r.label, "gray")
        draw.rectangle(r.bbox, outline=color, width=2)
        draw.text((r.bbox[0], r.bbox[1] - 12), f"{r.label} {r.confidence:.2f}", fill=color)

plt.figure(figsize=(12, 16))
plt.imshow(page)
plt.axis("off")
plt.show()

## 3. Test retrieval

In [ ]:
from src.retriever.baseline import SigLIPRetriever

retriever = SigLIPRetriever()

region_images = [r.image for r in regions]
embeddings = retriever.embed_images(region_images)
print(f"Embeddings shape: {embeddings.shape}")

query = "What is the main table in this document?"
results = retriever.retrieve(query, embeddings, top_k=3)
print(f"\nQuery: {query}")
for idx, score in results:
    r = regions[idx]
    print(f"  #{idx}: {r.label} (page {r.page_index}, score={score:.3f})")